## `modelling.ipynb: Step 2 the pipeline`

**Role**: Compute embeddings on Google Colab.           
**input**: `processed_df.pkl`.                      
**output**: `reviews_df.pkl` and `products_df.pkl`.

`Be aware to change the "DIRECTORY" variable (cell 3) with the right path of the folder on Google Drive where you uploaded the dataset!`

The `.pkl` format is sensitive to the Pandas version: the version used to create the file must match the one used to read it. Since Google Colab may ship a different Pandas version than the one on your local machine, run the cell below to install the exact version used during preprocessing.

In [1]:
!pip install pandas==3.0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 85.8 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pan

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Set DIRECTORY to the Google Drive folder where you uploaded processed_df.pkl.                   
*Examples*:
```python
# Dataset is directly in MyDrive root:
DIRECTORY = HERE / "drive" / "MyDrive"

# Dataset is inside a subfolder called "Data Science in Action":
DIRECTORY = HERE / "drive" / "MyDrive" / "Data Science in Action"
```

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

# --------------------------------------------------
# Paths
# --------------------------------------------------
HERE = Path.cwd()
DIRECTORY = HERE / "drive" / "MyDrive" / "Data Science in Action"  
INPUT_PATH  = DIRECTORY / "processed_df.pkl"
REVIEWS_OUTPUT_PATH  = DIRECTORY / "reviews_df.pkl"
PRODUCTS_OUTPUT_PATH = DIRECTORY / "products_df.pkl"


# --------------------------------------------------
# Model config
# --------------------------------------------------
MODEL_NAME = "intfloat/e5-base-v2" 

In [4]:
# --------------------------------------------------
# Device detection
# --------------------------------------------------
def get_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    elif torch.cuda.is_available():
        return "cuda"
    return "cpu"

In [ ]:
# --------------------------------------------------
# Load data
# --------------------------------------------------
def load_data(path: Path) -> pd.DataFrame:
    print(f"Loading data from: {path}")
    try:
        df = pd.read_pickle(path)
        print(f"Loaded {len(df):,} rows")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Failed to load reviews. Check if the path {path} is correct.") from e
    except (ModuleNotFoundError, ImportError) as e:
            raise ImportError(f"Version mismatch loading the dataset.\n"
                "Did you run the '!pip install pandas==3.0.1' cell?") from e
    return df

In [6]:
# --------------------------------------------------
# Compute embeddings
# --------------------------------------------------
def compute_embeddings(
    texts: list[str],
    model: SentenceTransformer,
    batch_size: int = 256
) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

In [7]:
# --------------------------------------------------
# Build a unified text profile for each product
# --------------------------------------------------
def build_profile(row) -> str:
    """
    Combines all product metadata fields (title, features, description,
    details, categories) into a single text string for embedding.
    """
    parts = []

    title = str(row.get("title_y", "") or "")
    if title:
        parts.append(title)

    features = row.get("features", [])
    if isinstance(features, list) and features:
        parts.append(". ".join(str(f) for f in features if f))

    description = row.get("description", [])
    if isinstance(description, list) and description:
        parts.append(". ".join(str(d) for d in description if d))

    details = row.get("details", {})
    if isinstance(details, dict) and details:
        parts.append(". ".join(f"{k}: {v}" for k, v in details.items()))

    categories = row.get("categories", [])
    if isinstance(categories, list) and categories:
        parts.append(", ".join(str(c) for c in categories if c))

    return " | ".join(p for p in parts if p.strip())

In [8]:
REVIEW_COLS  = ["rating", "title_x", "text", "user_id", "timestamp",
                "helpful_vote", "verified_purchase", "parent_asin", "review_embedding"]

PRODUCT_COLS = ["parent_asin", "title_y", "features", "description",
                "details", "categories", "store", "average_rating",
                "rating_number", "price", "product_embedding"]

def run_modelling(
    input_path: Path = INPUT_PATH,
    reviews_output_path: Path = REVIEWS_OUTPUT_PATH,
    products_output_path: Path = PRODUCTS_OUTPUT_PATH,
    model_name: str = MODEL_NAME
) -> tuple:

    df = load_data(input_path)

    device = get_device()
    print(f"Device: {device.upper()}")
    print(f"Loading model: {model_name}")
    model = SentenceTransformer(model_name, device=device)

    # 1. Review embeddings
    print("Encoding reviews...")
    review_texts = [
        "passage: " + (str(a) if a else "") + ". " + (str(b) if b else "")
        for a, b in zip(df["title_x"].fillna(""), df["text"].fillna(""))
    ]
    df["review_embedding"] = list(compute_embeddings(review_texts, model))

    # 2. Product profile embeddings (one per unique product)
    print("Building product profiles...")
    products = df.drop_duplicates("parent_asin").copy()
    profile_texts = [
        "passage: " + build_profile(row)
        for _, row in products.iterrows()
    ]
    products["product_embedding"] = list(compute_embeddings(profile_texts, model))

    # 3. Save two separate files
    reviews_df  = df[[c for c in REVIEW_COLS  if c in df.columns]].copy()
    products_df = products[[c for c in PRODUCT_COLS if c in products.columns]].copy()

    reviews_output_path.parent.mkdir(parents=True, exist_ok=True)
    reviews_df.to_pickle(reviews_output_path)
    print(f"Reviews saved:  {reviews_df.shape} → {reviews_output_path}")

    products_df.to_pickle(products_output_path)
    print(f"Products saved: {products_df.shape} → {products_output_path}")

    return reviews_df, products_df

In [9]:
run_modelling()

Loading data from: /content/drive/MyDrive/Data Science in Action/processed_df.pkl
Loaded 488,964 rows
Device: CUDA
Loading model: intfloat/e5-base-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

... (10 outputs left out)

Building product profiles...


Batches:   0%|          | 0/236 [00:00<?, ?it/s]

Reviews saved:  (488964, 9) → /content/drive/MyDrive/Data Science in Action/reviews_df.pkl
Products saved: (60272, 11) → /content/drive/MyDrive/Data Science in Action/products_df.pkl


(        rating                                            title_x  \
 0            4  12 mg is 12 on the periodic table people! Mg f...   
 1            5                 Save the lanet using less plastic.   
 2            5                                          Fantastic   
 3            4  It holds the water and makes bubbles.  That's ...   
 4            1                                         Not for me   
 ...        ...                                                ...   
 494116       5                                        Best brush!   
 494117       5                           It makes my skin softer.   
 494118       5                            This brush is perfect !   
 494119       5                                        Easy to use   
 494120       5       La comunicación con la empresa y ez perfecto   
 
                                                      text  \
 0       This review is more to clarify someone else’s ...   
 1       Love these easy multitask